<a href="https://colab.research.google.com/github/aly-elbana/Rag_Asses/blob/main/Rag_Project_NoteBook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Gemini Path

In [ ]:
!pip install -qU langchain langchain-core langchain-community langchain-google-genai langchain-huggingface langchain-chroma gradio pypdf sentence-transformers

In [ ]:
import os
from google.colab import userdata

os.environ["GOOGLE_API_KEY"] = userdata.get('GOOGLE_API_KEY') 


In [ ]:
import os
import shutil
import time
from pathlib import Path
from dotenv import load_dotenv
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings


BASE_DIR = Path.cwd()
timestamp = int(time.time())
DB_NAME = str(BASE_DIR / f"vector_db_{timestamp}")
# KNOWLEDGE_BASE = str(BASE_DIR / "PDFs") # Enable on Google colab
KNOWLEDGE_BASE = str(BASE_DIR.parent / "PDFs")

embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
load_dotenv(override=True)

def fetch_documents():
    kb_path = Path(KNOWLEDGE_BASE)
    if not kb_path.exists():
        print(f"Error: {KNOWLEDGE_BASE} not found")
        return []

    pdf_paths = list(kb_path.rglob("*.pdf"))
    documents = []

    print(f"Found {len(pdf_paths)} PDFs. Processing")

    for pdf_path in pdf_paths:
        try:
            doc_type = pdf_path.parent.name
            loader = PyPDFLoader(str(pdf_path))
            pages = loader.load()
            for page in pages:
                page.metadata["doc_type"] = doc_type
                page.metadata["source_file"] = pdf_path.name
                documents.append(page)

            print(f"Loaded: {pdf_path.name}")
        except Exception as e:
            print(f"Skipping {pdf_path.name}: {e}")

    return documents

def create_chunks(documents):
    if not documents:
        return []
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=700,
        chunk_overlap=100
    )
    return text_splitter.split_documents(documents)

def create_embeddings(chunks):
    if not chunks:
        print("Empty chunks list. Nothing to embed")
        return

    print(f"Generating embeddings for {len(chunks)} chunks")
    vectorstore = Chroma.from_documents(
        documents=chunks,
        embedding=embeddings,
        persist_directory=DB_NAME
    )

    print(f"Ingestion complete. DB stored at: {DB_NAME}")
    return vectorstore

docs = fetch_documents()
if docs:
    chunks = create_chunks(docs)
    create_embeddings(chunks)
else:
    print("No documents found. Check knowledge base folder")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 9801.78it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Found 4 PDFs. Processing
Loaded: lec 3.pdf
Loaded: lec 4.pdf
Loaded: lec 6.pdf
Loaded: lec 7 networks .pdf
Generating embeddings for 108 chunks
Ingestion complete. DB stored at: c:\Users\Aly Elbana\Desktop\Rag_Asses-main\Rag_Asses-main\src\vector_db_1773024197


In [5]:
import os
from pathlib import Path
# from google.colab import userdata Enable on google colab
import gradio as gr
from pathlib import Path


from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.messages import SystemMessage, HumanMessage, convert_to_messages
from langchain_core.documents import Document
from langchain_google_genai import ChatGoogleGenerativeAI


# os.environ["GOOGLE_API_KEY"] = userdata.get('GOOGLE_API_KEY')
os.environ["GOOGLE_API_KEY"] =  os.environ.get("GOOGLE_API_KEY")
 
MODEL = "gemini-2.5-flash"

embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
RETRIEVAL_K = 10

vectorstore = Chroma(persist_directory=DB_NAME, embedding_function=embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": RETRIEVAL_K})

llm = ChatGoogleGenerativeAI(model=MODEL, temperature=0)

SYSTEM_PROMPT = """
You are a knowledgeable, friendly assistant.
You are chatting with a user.
If relevant, use the given context to answer any question.
If you don't know the answer, say so.
Context:
{context}
"""

def fetch_context(question: str) -> list[Document]:
    return retriever.invoke(question)

def combined_question(question: str, history: list[dict] = []) -> str:
    prior = "\n".join(m["content"] for m in history if m["role"] == "user")
    return prior + "\n" + question

def answer_question(question: str, history: list[dict] = []) -> tuple[str, list[Document]]:
    combined = combined_question(question, history)
    docs = fetch_context(combined)

    context_text = "\n\n".join(doc.page_content for doc in docs)
    formatted_system_prompt = SYSTEM_PROMPT.format(context=context_text)

    messages = [SystemMessage(content=formatted_system_prompt)]
    messages.extend(convert_to_messages(history))
    messages.append(HumanMessage(content=question))

    response = llm.invoke(messages)
    return response.content, docs

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 10300.26it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


### Test Cell

In [6]:
chat_history = []

q1 = "what is the hub according to the docs"
ans1, docs1 = answer_question(q1, chat_history)
chat_history.append({"role": "user", "content": q1})
chat_history.append({"role": "assistant", "content": ans1})

q2 = "what was it's alternative according to lecture"
ans2, docs2 = answer_question(q2, chat_history)

print(ans2)

According to the lecture, the primary alternative to a hub is a **switch**.

The documents state:
*   "If you swapped the hub for a switch, you would have four collision domains (one per port used)."
*   "We replace the main hub with switch, because hubs don’t segment a network, they just connect network segments together."

Switches are presented as a more efficient alternative because they build a list of which PCs are connected to which ports, allowing them to forward traffic only to the intended recipient, unlike hubs which send data to all ports.


In [7]:
import gradio as gr
from pathlib import Path

def chat_wrapper(message, history):
    formatted_history = []

    for item in history:
        if isinstance(item, (list, tuple)) and len(item) >= 2:
            human, ai = item[0], item[1]
            if human:
                formatted_history.append({"role": "user", "content": str(human)})
            if ai:
                clean_ai = str(ai).split("\n\n---")[0]
                formatted_history.append({"role": "assistant", "content": clean_ai})

        elif isinstance(item, dict) and "role" in item and "content" in item:
            role = item["role"]
            content = str(item["content"])
            if role == "assistant" and "\n\n---" in content:
                content = content.split("\n\n---")[0]
            formatted_history.append({"role": role, "content": content})

        elif hasattr(item, "role") and hasattr(item, "content"):
            role = item.role
            content = str(item.content)
            if role == "assistant" and "\n\n---" in content:
                content = content.split("\n\n---")[0]
            formatted_history.append({"role": role, "content": content})

    answer, docs = answer_question(message, formatted_history)

    source_list = "\n\n---\n**Sources used:**"
    unique_sources = set()
    for doc in docs:
        source_name = doc.metadata.get('source', 'Unknown Document')
        unique_sources.add(Path(str(source_name)).name)

    for s in unique_sources:
        source_list += f"\n- {s}"

    return f"{answer}{source_list}"

demo = gr.ChatInterface(
    fn=chat_wrapper,
    title="Gemini RAG Assistant",
    description="Ask questions about your documents.",
)

if __name__ == "__main__":
    demo.launch(debug=True)

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


Keyboard interruption in main thread... closing server.


# Open Weights Models (HuggingFace)


In [ ]:
import os
import shutil
from pathlib import Path
from dotenv import load_dotenv
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings

BASE_DIR = Path.cwd()
DB_NAME = str(BASE_DIR / "vector_db")
KNOWLEDGE_BASE = str(BASE_DIR / "PDFs")

embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
load_dotenv(override=True)

def fetch_documents():
    kb_path = Path(KNOWLEDGE_BASE)
    if not kb_path.exists():
        print(f"Error: {KNOWLEDGE_BASE} not found")
        return []

    pdf_paths = list(kb_path.rglob("*.pdf"))
    documents = []

    print(f"Found {len(pdf_paths)} PDFs. Processing")

    for pdf_path in pdf_paths:
        try:
            doc_type = pdf_path.parent.name
            loader = PyPDFLoader(str(pdf_path))
            pages = loader.load()
            for page in pages:
                page.metadata["source"] = pdf_path.name
                page.metadata["doc_type"] = doc_type
                documents.append(page)

            print(f"Loaded: {pdf_path.name}")
        except Exception as e:
            print(f"Skipping {pdf_path.name}: {e}")

    return documents

def create_chunks(documents):
    if not documents:
        return []
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=700,
        chunk_overlap=100
    )
    return text_splitter.split_documents(documents)

def create_embeddings(chunks):
    if not chunks:
        print("Empty chunks list. Nothing to embed")
        return

    print(f"Generating embeddings for {len(chunks)} chunks")
    vectorstore = Chroma.from_documents(
        documents=chunks,
        embedding=embeddings,
        persist_directory=DB_NAME
    )

    print(f"Ingestion complete. DB stored at: {DB_NAME}")
    return vectorstore

docs = fetch_documents()
if docs:
    chunks = create_chunks(docs)
    create_embeddings(chunks)
else:
    print("No documents found. Check knowledge base folder")

In [ ]:
import os
from pathlib import Path
import gradio as gr
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_huggingface import HuggingFacePipeline
from langchain_core.documents import Document
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
import torch

MODEL_ID = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    device_map="auto",
    torch_dtype=torch.float16
)

pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=512,
    temperature=0.7,
    do_sample=True,
    pad_token_id=tokenizer.eos_token_id
)

llm = HuggingFacePipeline(pipeline=pipe)

DB_NAME = str(Path("vector_db"))

embeddings = HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2"
)

RETRIEVAL_K = 4

vectorstore = Chroma(
    persist_directory=DB_NAME,
    embedding_function=embeddings
)

retriever = vectorstore.as_retriever(
    search_kwargs={"k": RETRIEVAL_K}
)

SYSTEM_PROMPT = """
You are a knowledgeable, friendly assistant.
You are chatting with a user.

If relevant, use the given context to answer any question.
If you don't know the answer, say so.

Context:
{context}
"""

def fetch_context(question: str) -> list[Document]:
    return retriever.invoke(question)

def combined_question(question: str, history: list[dict] = []) -> str:
    prior = "\n".join(
        m["content"]
        for m in history
        if m["role"] == "user"
    )
    return prior + "\n" + question


In [ ]:
import re
import gradio as gr
from pathlib import Path

def clean_llm_output(text: str) -> str:
    """
    Strips out <think> blocks and cleans up trailing whitespace.
    Adjust the regex if your specific model uses different tags (e.g., [THOUGHT]).
    """
    # Remove everything between <think> and </think> (DOTALL makes '.' match newlines)
    cleaned_text = re.sub(r'<think>.*?</think>', '', text, flags=re.DOTALL)
    
    # Optional: Remove loose '### Thinking ###' headers if the model uses them
    cleaned_text = re.sub(r'### Thinking.*?\n', '', cleaned_text, flags=re.IGNORECASE)
    
    return cleaned_text.strip()

def answer_question(question: str, history: list[dict] = []):
    prior_questions = "\n".join(
        m["content"] for m in history if m["role"] == "user"
    )

    combined_query = f"""
Conversation history:
{prior_questions}

Current question:
{question}
"""

    docs = retriever.invoke(combined_query)

    seen = set()
    unique_docs = []
    for doc in docs:
        content = doc.page_content.strip()
        if content not in seen:
            seen.add(content)
            unique_docs.append(doc)

    context_text = "\n\n".join(doc.page_content for doc in unique_docs)

    system_prompt = f"""
You are a knowledgeable, friendly assistant.
Answer concisely using the given context.
If the answer is unknown, say so. Do not output your internal thinking process.

Context:
{context_text}
"""

    chat_history_text = ""
    for msg in history:
        if msg["role"] == "user":
            chat_history_text += f"User: {msg['content']}\n"
        elif msg["role"] == "assistant":
            chat_history_text += f"Assistant: {msg['content']}\n"

    final_prompt = f"""
### System
{system_prompt}

### Conversation
{chat_history_text}

### User
{question}

### Assistant
"""

    # Get response from LLM
    raw_response = llm.invoke(final_prompt)
    
    # Extract text
    response_text = raw_response.content if hasattr(raw_response, "content") else str(raw_response)
    
    # --- NEW FIX: Strip out the echoed prompt ---
    # Split the text at "### Assistant" and take only the last part (the generated answer)
    if "### Assistant" in response_text:
        response_text = response_text.split("### Assistant")[-1].strip()
    
    # Clean the response to hide the thinking process
    final_answer = clean_llm_output(response_text)
    
    return final_answer, unique_docs

def chat_wrapper(message, history):
    formatted_history = []

    # Parse Gradio history (handles both old tuples and new dicts/objects formats)
    for item in history:
        if isinstance(item, (list, tuple)) and len(item) >= 2:
            human, ai = item[0], item[1]
            if human:
                formatted_history.append({"role": "user", "content": str(human)})
            if ai:
                # Clean up the AI's past responses so thoughts don't leak into context
                clean_ai = clean_llm_output(str(ai).split("\n\n---")[0])
                formatted_history.append({"role": "assistant", "content": clean_ai})
                
        elif isinstance(item, dict) and "role" in item and "content" in item:
            role = item["role"]
            content = str(item["content"])
            if role == "assistant":
                content = clean_llm_output(content.split("\n\n---")[0])
            formatted_history.append({"role": role, "content": content})
            
        elif hasattr(item, "role") and hasattr(item, "content"):
            role = item.role
            content = str(item.content)
            if role == "assistant":
                content = clean_llm_output(content.split("\n\n---")[0])
            formatted_history.append({"role": role, "content": content})

    # Get the clean answer and docs
    answer, docs = answer_question(message, formatted_history)

    # Format the sources list
    source_list = "\n\n---\n**Sources used:**"
    unique_sources = set()
    for doc in docs:
        source_name = doc.metadata.get("source", "Unknown Document")
        unique_sources.add(Path(str(source_name)).name)
        
    for s in unique_sources:
        source_list += f"\n- {s}"

    # Return final appended string for Gradio UI
    return f"{answer}{source_list}"

demo = gr.ChatInterface(
    fn=chat_wrapper,
    title="TinyLlama RAG Assistant",
    description="Ask questions about your documents."
)

if __name__ == "__main__":
    # Make sure your 'retriever' and 'llm' objects are defined before this point!
    demo.launch(debug=True)